# KDE-BDPC — 100-seed Reacher cluster sweep

**Latest protocol:** K = **2, 4, 8, 16, 32, 48, 64, 80, 96**; development seeds **211–310**; **T_sim = 300**, **T_ini = 2**, **T_f = 1**, **p = 8**. This is **900 rollout attempts**, at most **270,000 control steps** excluding warm-up.

The controller implements the paper's Eq. (34), with affine cluster representations and automatic online rank selection in Eq. (24). Only cluster count varies; the other settings below are fixed. This is development screening, not held-out evaluation.

Use the batch controls below. All action switches default to **False**; executing the notebook in this state only displays settings and saved results.


In [1]:
from pathlib import Path
import sys, importlib
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents, Path('/Users/kumiko/Projects/atlas_deepc/code')]
            if (p/'src/comparison_methods/kde_bdpc/batches.py').is_file())
if str(ROOT/'src') not in sys.path:
    sys.path.insert(0, str(ROOT/'src'))
from comparison_methods.kde_bdpc import batches
importlib.reload(batches)

config = batches.study.Config()
config.validate()
T_SIM = batches.STEPS
assert (config.past, config.future, T_SIM) == (2, 1, 300)
assert batches.CLUSTERS == (2, 4, 8, 16, 32, 48, 64, 80, 96)
assert batches.SEEDS == tuple(range(211, 311))
print('External-drive output:', batches.OUTPUT)


External-drive output: /Volumes/Crucial X10/atlas_deepc_data/experiments/reacher/kde_bdpc_eq34_l3_clusters9_dev100_t300_v1


## Current parameters

The table shows the settings actually used by the batch runner, not unused tuning proposals. Equation references identify each parameter's role, not an endorsement of our numerical values by the authors. Changing a fixed controller parameter requires a matching runner/protocol update and a new results directory.


In [2]:
PAPER_PARAMETER_REFERENCES = {
    'clusters': ('N_c', 'Eqs. (9a)-(9b), Sec. 2.3, p. 3', 'Number of clusters / NMF columns.'),
    'nmf_seed': ('Not a paper parameter', 'No equation; implementation of Eq. (9)', 'Our random-initialization seed, not a paper tuning parameter.'),
    'lambda_view1': ('lambda_1 (v=1)', 'Eq. (9a), p. 3', 'Consensus penalty for the Euclidean-distance view.'),
    'lambda_view2': ('lambda_2 (v=2)', 'Eq. (9a), p. 3', 'Consensus penalty for the subspace-distance view.'),
    'svd_rtol': ('No prescribed cutoff symbol', 'Eqs. (5)-(6) and intervening text, p. 3; Sec. 2.3, p. 4', 'Our offline relative cutoff; NOT the online rank rule in Eq. (24).'),
    'beta_kde': ('beta_kde; B = beta_kde I', 'Eqs. (15)-(16); definition after Eq. (16), p. 4', 'Shared Gaussian kernel covariance/bandwidth multiplier.'),
    'beta_bound': ('beta_bound', 'Set G-bar for Eq. (18), defined after Eq. (25), p. 5; Eqs. (28c), (34c)', 'Radius ||g||_2 <= beta_bound for interpolated subspaces.'),
    'beta_s': ('beta_s', 'Eqs. (27c), (29), pp. 6-7', 'Source-cluster density constraint and target-cluster stopping threshold.'),
    'lambda_epsilon': ('lambda_epsilon', 'Eq. (34a), p. 8', 'Squared past-mismatch penalty.'),
    'q_joint': ('Entries of Lambda_y', 'Eq. (34a), p. 8', 'Our Reacher weights on the four joint sine/cosine output channels.'),
    'q_dq': ('Entries of Lambda_y', 'Eq. (34a), p. 8', 'Our Reacher weights on the two joint-velocity output channels.'),
    'q_xy': ('Entries of Lambda_y', 'Eq. (34a), p. 8', 'Our Reacher weights on the two fingertip-position output channels.'),
    'r': ('Lambda_u = r I', 'Eq. (34a), p. 8', 'Our scalar parameterization of the input-reference tracking weight.'),
}
parameter_rows = []
for name, (symbol, equation, meaning) in PAPER_PARAMETER_REFERENCES.items():
    value = ', '.join(map(str, batches.CLUSTERS)) if name == 'clusters' else str(getattr(config, name))
    parameter_rows.append({
        'Parameter': name, 'Current value(s)': value,
        'Paper symbol / role': symbol, 'Paper equation / location': equation,
        'Interpretation': meaning,
    })
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', 100):
    display(pd.DataFrame(parameter_rows))
print('Other fixed settings: warm-up =', config.warmup,
      '; J = 0; solver = CLARABEL; initial/reference-cluster policy =', config.same_initial_cluster)
print('Online interpolated rank: automatic Eq. (24), admissible candidate dimensions 7–29.')


,Parameter,Current value(s),Paper symbol / role,Paper equation / location,Interpretation
0,clusters,"2, 4, 8, 16, 32, 48, 64, 80, 96",N_c,"Eqs. (9a)-(9b), Sec. 2.3, p. 3",Number of clusters / NMF columns.
1,nmf_seed,42,Not a paper parameter,No equation; implementation of Eq. (9),"Our random-initialization seed, not a paper tuning parameter."
2,lambda_view1,0.1,lambda_1 (v=1),"Eq. (9a), p. 3",Consensus penalty for the Euclidean-distance view.
3,lambda_view2,0.1,lambda_2 (v=2),"Eq. (9a), p. 3",Consensus penalty for the subspace-distance view.
4,svd_rtol,0.01,No prescribed cutoff symbol,"Eqs. (5)-(6) and intervening text, p. 3; Sec. 2.3, p. 4",Our offline relative cutoff; NOT the online rank rule in Eq. (24).
5,beta_kde,1.5,beta_kde; B = beta_kde I,"Eqs. (15)-(16); definition after Eq. (16), p. 4",Shared Gaussian kernel covariance/bandwidth multiplier.
6,beta_bound,0.05,beta_bound,"Set G-bar for Eq. (18), defined after Eq. (25), p. 5; Eqs. (28c), (34c)",Radius ||g||_2 <= beta_bound for interpolated subspaces.
7,beta_s,0.8,beta_s,"Eqs. (27c), (29), pp. 6-7",Source-cluster density constraint and target-cluster stopping threshold.
8,lambda_epsilon,1000.0,lambda_epsilon,"Eq. (34a), p. 8",Squared past-mismatch penalty.
9,q_joint,4000.0,Entries of Lambda_y,"Eq. (34a), p. 8",Our Reacher weights on the four joint sine/cosine output channels.


Other fixed settings: warm-up = 12 ; J = 0; solver = CLARABEL; initial/reference-cluster policy = reject
Online interpolated rank: automatic Eq. (24), admissible candidate dimensions 7–29.


## Run a batch

1. Connect Crucial X10. The launcher refuses to overlap the earlier one-seed screen or another batch.
2. Start with **two workers** and **seeds 211–220**. With nine cluster counts, that is up to **90 new rollout attempts**. Set `RUN_BATCH = True` in the controls cell, then run the launch cell once; set the flag back to False afterward.
3. For the next block, change the endpoints to **221, 230**. Repeating a block skips saved attempts. To take the next pending seeds automatically, set both endpoints to `None`. `BATCH_SIZE` limits new attempts **per cluster count**, not total.
4. Refresh the results cell to monitor the detached job. To stop gently, set `STOP_BATCH = True` and `RUN_BATCH = False`, then run the launch cell. Active workers finish the current rollout or offline preparation; queued jobs are not started. Once all workers stop, use `CLEAR_STOP_REQUEST = True` once before resuming.

Parallelism is across cluster counts; each worker runs its seeds sequentially with one BLAS thread. Verified offline models/graphs may be reused from the earlier screen, but **no 150-step rollout traces or statistics are imported**. Setup rejections and solver failures remain recorded attempts; an interrupted seed without a saved final trace restarts from the beginning.

Keep the computer awake and the external drive connected while jobs are active. **Parallel timings can be affected by contention:** use a separate serial evaluation for timing comparisons with Atlas.


In [3]:
assert batches.STEPS == T_SIM == 300
BATCH_OUTPUT = batches.OUTPUT
CLUSTERS_TO_RUN = list(batches.CLUSTERS)
PARALLEL_WORKERS = 6
BATCH_SIZE = 50                 # Maximum NEW attempts PER cluster per invocation.
BLOCK_START, BLOCK_STOP = 211, 310  # Next block: 221, 230. Both None = automatic resume.
RUN_BATCH = False
STOP_BATCH = False
CLEAR_STOP_REQUEST = False

batch_args = batches.parser().parse_args([
    '--output', str(BATCH_OUTPUT), '--workers', str(PARALLEL_WORKERS),
    '--batch-size', str(BATCH_SIZE), '--clusters', *map(str, CLUSTERS_TO_RUN),
    *([] if BLOCK_START is None and BLOCK_STOP is None else
      ['--block-start', str(BLOCK_START), '--block-stop', str(BLOCK_STOP)]),
])
batches.validate(batch_args)
scope = batches.requested_seeds(BLOCK_START, BLOCK_STOP)
print('Full panel: 9 × 100 × 300 steps = 900 rollout attempts, at most 270,000 control steps.')
print(f'This invocation: up to {min(BATCH_SIZE,len(scope))*len(CLUSTERS_TO_RUN)} NEW attempts; '
      f'{PARALLEL_WORKERS} parallel cluster workers.')
print('Seeds in scope:', scope[0], 'through', scope[-1])
print('Already saved 300-step-protocol attempts are skipped; 150-step traces are never imported.')
print('Results directory:', BATCH_OUTPUT)


Full panel: 9 × 100 × 300 steps = 900 rollout attempts, at most 270,000 control steps.
This invocation: up to 450 NEW attempts; 6 parallel cluster workers.
Seeds in scope: 211 through 310
Already saved 300-step-protocol attempts are skipped; 150-step traces are never imported.
Results directory: /Volumes/Crucial X10/atlas_deepc_data/experiments/reacher/kde_bdpc_eq34_l3_clusters9_dev100_t300_v1


In [4]:
# Only this cell starts/stops a batch, and only with an explicit flag.
import os, subprocess, shlex
from datetime import datetime, timezone

if RUN_BATCH and STOP_BATCH:
    raise ValueError('Choose either RUN_BATCH or STOP_BATCH, not both.')
if CLEAR_STOP_REQUEST:
    if BATCH_OUTPUT.exists():
        with batches.lock_file(BATCH_OUTPUT/'batch.lock'):
            flag = BATCH_OUTPUT/'STOP_AFTER_CURRENT'
            if flag.exists():
                flag.unlink()  # Only the explicit stop marker; never result files.
        print('Cleared stop request; saved results unchanged.')
if STOP_BATCH:
    if not BATCH_OUTPUT.exists():
        raise RuntimeError('No batch output directory exists yet.')
    (BATCH_OUTPUT/'STOP_AFTER_CURRENT').touch()
    print('Stop requested. Wait for active workers to finish before disconnecting the drive.')
if RUN_BATCH:
    batches.validate(batch_args)
    if not batches.study.core.SSD.is_dir():
        raise FileNotFoundError('Connect Crucial X10')
    batches.require_idle_screen(batch_args.reuse_screen)
    BATCH_OUTPUT.mkdir(parents=True, exist_ok=True)
    with batches.lock_file(BATCH_OUTPUT/'batch.lock'):
        if (BATCH_OUTPUT/'STOP_AFTER_CURRENT').exists():
            raise RuntimeError('Clear the stop request explicitly before resuming.')
    command = [sys.executable, '-u', '-m', 'comparison_methods.kde_bdpc.batches',
               '--output', str(BATCH_OUTPUT), '--workers', str(PARALLEL_WORKERS),
               '--batch-size', str(BATCH_SIZE), '--clusters', *map(str, CLUSTERS_TO_RUN)]
    if BLOCK_START is not None:
        command += ['--block-start', str(BLOCK_START), '--block-stop', str(BLOCK_STOP)]
    env = os.environ.copy()
    env['PYTHONPATH'] = str(ROOT/'src')
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    with (BATCH_OUTPUT/f'launcher_{stamp}.log').open('a') as log:
        process = subprocess.Popen(command, cwd=ROOT, env=env, stdin=subprocess.DEVNULL,
                                   stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    batches.write(BATCH_OUTPUT/f'launch_{stamp}.json', {
        'pid': process.pid, 'command': command, 'cwd': str(ROOT), 'started_at_utc': stamp})
    print('Batch launcher PID:', process.pid)
    print('Command:', shlex.join(command))
    print('Startup/manager log:', BATCH_OUTPUT/f'launcher_{stamp}.log')
    print('Worker logs: kXX_batch.log in the result directory. Refresh the status cell below.')
if not any((RUN_BATCH, STOP_BATCH, CLEAR_STOP_REQUEST)):
    print('No batch launched or stopped. All action switches are off.')


Batch launcher PID: 44525
Command: /Users/kumiko/miniconda3/envs/gym-robotics-mujoco/bin/python -u -m atlas_deepc.benchmarks.kde_bdpc_paper_batches --output '/Volumes/Crucial X10/atlas_deepc_data/experiments/reacher/kde_bdpc_eq34_l3_clusters9_dev100_t300_v1' --workers 6 --batch-size 50 --clusters 2 4 8 16 32 48 64 80 96 --block-start 211 --block-stop 310
Startup/manager log: /Volumes/Crucial X10/atlas_deepc_data/experiments/reacher/kde_bdpc_eq34_l3_clusters9_dev100_t300_v1/launcher_20261101T204031354908Z.log
Worker logs: kXX_batch.log in the result directory. Refresh the status cell below.


## Current 300-step results

Rerun the following cell to refresh. **K requested** is the configured number of clusters; **K active** is the number of nonempty clusters after hard assignment. Empty clusters are omitted as an explicit implementation convention, not an author-specified repair rule.

Saved attempts include setup rejections and aborted rollouts; **full rollouts** reached all 300 steps. Final-100 error summarizes completed rollouts only. Missing metrics are unavailable, not zero. Old screen results are not displayed or pooled here.


In [10]:
# Read-only refresh: no simulations and no directory creation.
panel_rows = batches.snapshot(BATCH_OUTPUT)
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(pd.DataFrame(panel_rows).rename(columns={
        'clusters': 'K requested', 'active_clusters': 'K active',
        'processed': 'Saved attempts / 100', 'complete': 'Full rollouts',
        'setup_rejections': 'Setup rejections', 'other_aborts': 'Other aborts',
        'blocked': 'Blocked by offline rejection', 'pending': 'Pending',
        'final100_error_mm': 'Median final-100 error (mm)',
        'median_online_ms': 'Median online time (ms; see timing caveat)'}))
print('Batch manager:', batches.read(BATCH_OUTPUT/'batch_progress.json') or 'Not started')
print('Full panel:', sum(r['processed'] for r in panel_rows), '/ 900 saved attempts; ',
      sum(r['blocked'] for r in panel_rows), 'blocked by offline rejection.')


,K requested,K active,state,Saved attempts / 100,Full rollouts,successes,Setup rejections,Other aborts,Blocked by offline rejection,Pending,Median final-100 error (mm),Median online time (ms; see timing caveat)
0,2,2.0,batch_complete,50,8,0.0,41,1,0,50,209.035091,24.085249
1,4,3.0,batch_complete,50,17,0.0,30,3,0,50,229.755448,27.370625
2,8,5.0,running,43,31,0.0,10,2,0,57,251.854035,40.217167
3,16,13.0,running,11,11,0.0,0,0,0,89,274.236718,477.616209
4,32,22.0,running,22,21,0.0,0,1,0,78,272.856234,254.035334
5,48,31.0,running,3,3,0.0,0,0,0,97,271.861407,1398.283104
6,64,38.0,running,2,2,0.0,0,0,0,98,214.805316,1088.172312
7,80,45.0,running,1,1,0.0,0,0,0,99,52.640902,2643.813666
8,96,NaN,not_started,0,0,NaN,0,0,0,100,NaN,NaN


Batch manager: {'state': 'running', 'batch_id': '20261101T204032446862Z', 'active': {'8': 44535, '16': 44536, '32': 44537, '48': 44538, '64': 44756, '80': 45308}, 'queued': [96], 'failures': []}
Full panel: 182 / 900 saved attempts;  0 blocked by offline rejection.


## Equation-to-code specification

| Paper | Implementation |
|---|---|
| §2.3, Eqs. (5)–(6), (14) | Cluster center is the mean of all length-L segments. Fit an orthonormal basis to deviations from that center. Represent a cluster as **c_i + F_i g**, not F_i g through the origin. |
| Eqs. (15), (17) | Gaussian KDE on all standardized cluster segments; normalized membership probabilities. |
| Theorem 3, Eq. (24) | Membership-weighted projector eigendecomposition; choose the interpolated dimension by weighted chordal distance. No fixed Atlas rank. |
| Eq. (18) | Online affine offset is the latest measured length-L trajectory; interpolated basis is recomputed at that point. |
| Eqs. (27)–(29), Algorithms 1–3 | Connections follow the printed constrained objectives and terminate on the density-ratio test. Nonconvergent/stalled connections fail explicitly. |
| Eqs. (30)–(32), Algorithm 4 | Entropy edge weights; minimize **mean**, not total, path weight. Certified simple-path optimization. |
| Eq. (33) | Track the next cluster's center, or the final reference when the next vertex is its cluster. |
| Eq. (34), Algorithm 5 | Squared output/input-reference tracking over L samples; squared mismatch on both inputs and outputs for the L-1 overlap; coefficient ball; one new input/output pair. |


## Implementation notes

- **Offline versus online rank:** `svd_rtol` is our numerical choice for the offline low-rank approximation around Eqs. (5)–(6). The online interpolated rank is selected automatically by Theorem 3 / Eq. (24).
- **Affine controller:** the latest measured length-L window is the online offset. Eq. (34) tracks input/output references over the shifted window, penalizes mismatch on its L−1 overlapping pairs, and constrains the coefficient norm. No slack is added to the final predicted pair; only its input is applied. Reacher uses J = 0.
- **Connection objective:** printed Eqs. (27a)/(28a) sum Gaussian exponents, whereas Eq. (26) describes maximizing KDE; these are not generally equivalent. We follow the printed objectives and retain the actual KDE-ratio tests. The Eq. (27) SLSQP solve is local, not globally certified.
- **Graph conventions:** directed connections, simple paths, and minimum mean edge cost are explicit conventions. Connections stop on the density-ratio condition or detected stagnation; no approximate route fallback is used.
- **Initial/reference cluster:** the strict implementation rejects cases outside the different-cluster setup in Section 7.1. These are reported separately from completed tracking trials.
- **Reacher choices:** standardized channels, fixed development references, p = 8, and L = 3 (two overlapping pairs plus one predicted pair). Matching Atlas's T_ini and simulation length does not establish the paper's lag/local-model assumptions or match Atlas's prediction horizon.
- **Physical actuation:** the optimizer uses the coefficient ball, without added torque constraints; applied commands are clipped to the environment's bounds. Both commanded and applied inputs are saved.

This is an implementation of the printed equations with disclosed choices, **not an author-verified reproduction**. All 300-step traces, predictions, diagnostic quantities, timing, and source/config/data provenance remain on Crucial X10.

Reference: Han et al., *Big data-driven predictive control for nonlinear systems based on kernel density estimation of data trajectories*, Computers & Chemical Engineering 207 (2026), 109565. [Paper DOI](https://doi.org/10.1016/j.compchemeng.2026.109565).
